# `sample_van_der_corput()`

The analysis function `nematics3d.sample_van_der_corput()` generates the first points of the base-2 van der Corput sequence on the half-open unit interval $[0,1)$. The sequence is deterministic and progressively fills gaps at finer binary scales, making it useful when a fixed ordering of well-distributed sample positions is more useful than an evenly spaced grid chosen for one particular sample count.

The first values are $0$, $1/2$, $1/4$, $3/4$, $1/8$, $5/8$, $3/8$, and $7/8$. Because the ordering is intrinsic to the sequence, requesting more points extends the existing prefix rather than moving earlier sample positions.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example

Requesting eight samples returns the first eight entries of the sequence.


In [2]:
samples = n3d.sample_van_der_corput(8)
print(samples)

[0.    0.5   0.25  0.75  0.125 0.625 0.375 0.875]


## Inputs and outputs

The public signature is:

```python
sample_van_der_corput(num)
```

`num` is the number of requested samples and must be a non-negative integer-valued real number. Boolean, negative, `NaN`, and infinite inputs are rejected.

The return value is a one-dimensional floating-point `NumPy` array of length `num`. Every value lies in $[0,1)$; in particular, the endpoint $1$ is not part of the van der Corput sequence. `num=0` returns an empty floating-point array.


## Progressive sampling

A useful property of the sequence is prefix stability. Increasing the requested sample count preserves every sample that was already present. This differs from replacing an $N$-point uniform grid with an $(N+1)$-point uniform grid, which generally moves all interior points.


In [3]:
short = n3d.sample_van_der_corput(4)
long = n3d.sample_van_der_corput(12)

print("short sequence:", short)
print("long prefix matches:", np.array_equal(short, long[: len(short)]))

short sequence: [0.   0.5  0.25 0.75]
long prefix matches: True


## Details

The base-2 van der Corput sequence is the radical-inverse sequence in base 2. If a non-negative integer $n$ has binary expansion

$$n=\sum_{k=0}^{m} a_k 2^k, \qquad a_k\in\{0,1\},$$

then its van der Corput value is obtained by reflecting those binary digits across the radix point:

$$\phi_2(n)=\sum_{k=0}^{m} a_k 2^{-(k+1)}.$$

For example, $6=(110)_2$, so reversing the binary digits after the radix point gives $0.011_2=3/8$.

`Nematics3D` generates the sequence recursively from the identities $\phi_2(2n)=\phi_2(n)/2$ and $\phi_2(2n+1)=1/2+\phi_2(n)/2$. The implementation fills complete binary levels at a time with `NumPy`, avoiding a separate binary-digit loop for every requested sample.


## Possible issues

The sequence is designed for progressive low-discrepancy sampling, not for including both endpoints of the unit interval. If an application specifically requires $0$ and $1$ to appear first, a different sampling rule should be used rather than modifying the van der Corput definition.

The current public function intentionally implements only base 2. Higher-dimensional quasi-Monte Carlo sampling or arbitrary-base radical-inverse sequences should use a dedicated method rather than overloading this small helper.
